## Assignment 3: Fine-Tuning a Language Model for Text Classification

### Exercise 1: Introduction to Fine-Tuning a BERT Model for Text Classification
In this exercise, you will gain practical experience in fine-tuning a pre-trained language model for a text classification task. You will use the `bert-base-cased` model and the `yelp_review_full` dataset, which contains Yelp business reviews classified into five-star ratings. You are also free to choose a different pre-trained model or dataset if you prefer, but in that case, you will need to adapt the provided code accordingly.

Your goal is to fine-tune the model to predict the rating based on the text of the review. You are required to complete the following tasks:
- Load and prepare a text dataset for fine-tuning.
- Fine-tune a pre-trained language model for text classification.
- Evaluate the model's performance.
- Reflect on the model's strengths and weaknesses.

Example code snippets are provided below to help you complete the tasks. You can use them as a starting point for your solution. Alternatively, you are welcome to use other libraries and resources if you prefer.

### Exercise 2: Fine-Tuning a BERT Model for Text Classification with Your Own Dataset
In this exercise, you will use your own classified State of the Union dataset to fine-tune a BERT model for text classification. You can use the `bert-base-cased` model or select a different pre-trained model if you prefer. The goal is to fine-tune the model to predict the sentiment you assigned to paragraphs (or other subsets of the speeches in the first assignment) based on the text of the speech.

Although we do not provide specific code for this exercise, you should be able to reuse much of the code from Exercise 1 with some modifications. Here are some suggestions to guide your approach:
- **Dataset Creation**: Create a dataset with text samples and corresponding sentiment labels. This can be a binary classification task (e.g., positive vs. negative sentiment) or a multi-class classification task (e.g., positive vs. neutral vs. negative sentiment). You will need to choose appropriate cutoffs for the sentiment scores to create the labels.
- **Fine-Tuning**: Fine-tune a pre-trained language model for text classification. You can adapt the code from the previous exercise to work with your dataset and classification task.
- **Evaluation**: Evaluate the model's performance using the same evaluation metrics from Exercise 1 or other suitable metrics. How well does this approach work for your dataset compared to the simple approach you used in the first assignment? If you did not use a simple approach in the first assignment, you should do one now to compare to the results here. **Make sure you are using the same data for the different approaches.** Use e.g., a dictionary based approach, a boolean approach, or a simple machine learning model like the Vader sentiment analysis model. 
- **Analysis**: Discuss the strengths and weaknesses of your model. How well does it perform on the sentiment classification task? What factors influence its performance? Do you see any benefits or limitations of using a pre-trained language model for this task over a simpler approach?


**Note 1**: *This exercise, especially Exercise 1, can be computationally demanding. During the lecture "Practical Day 2," we will discuss how to potentially run this on a server to make it feasible to include more training data.*

**Note 2**: *The code provided below is based on the Hugging Face documentation. You can find more details here: [Hugging Face Transformers Training Documentation](https://huggingface.co/docs/transformers/en/training).*

In [1]:
# Import the load_dataset function from the datasets library to load and work with text datasets (you may need to install the library first)
from datasets import load_dataset

# Load the 'yelp_review_full' dataset, which contains Yelp business reviews with five-star ratings
dataset = load_dataset("yelp_review_full")

# Access and display the 100th sample from the 'train' split of the dataset
dataset["train"][100]


/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'label': 0,
 'text': 'My expectations for McDonalds are t rarely high. But for one to still fail so spectacularly...that takes something special!\\nThe cashier took my friends\'s order, then promptly ignored me. I had to force myself in front of a cashier who opened his register to wait on the person BEHIND me. I waited over five minutes for a gigantic order that included precisely one kid\'s meal. After watching two people who ordered after me be handed their food, I asked where mine was. The manager started yelling at the cashiers for \\"serving off their orders\\" when they didn\'t have their food. But neither cashier was anywhere near those controls, and the manager was the one serving food to customers and clearing the boards.\\nThe manager was rude when giving me my order. She didn\'t make sure that I had everything ON MY RECEIPT, and never even had the decency to apologize that I felt I was getting poor service.\\nI\'ve eaten at various McDonalds restaurants for over 30 years. 

In [2]:
# Import the AutoTokenizer class from the transformers library for loading a tokenizer (you need to install the tokenizer library first)
from transformers import AutoTokenizer

# Load a pre-trained tokenizer from the 'google-bert/bert-base-cased' model
# This tokenizer will be used to tokenize the text data for input into the model
# We will use this model for fine-tuning on the Yelp review dataset, if you have a different model in mind, you can replace it here
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

# Define a function to tokenize input examples from the dataset
def tokenize_function(examples):
    # Tokenize the 'text' field of the examples with padding and truncation
    # 'padding="max_length"' pads the input sequences to the model's maximum length
    # 'truncation=True' ensures that input sequences longer than the maximum length are truncated
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply the tokenize_function to the entire dataset using the map() method
# 'batched=True' processes examples in batches for faster computation
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 50000/50000 [00:09<00:00, 5296.72 examples/s]


## Train with PyTorch 

In [3]:

import sys
print(sys.executable)

# Import PyTorch (needs to be installed)
import torch
print(torch.__version__)


import torch
print(torch.__version__)
print(torch.cuda.is_available())  # Optional: check if CUDA is available (if needed)


/Users/mr.engelschion/anaconda3/envs/assignment3/bin/python
2.5.1
2.5.1
False


In [4]:
# Remove the text column because the model does not accept raw text as an input
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Rename the label column to labels because the model expects the argument to be named labels
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set the format of the dataset to return PyTorch tensors instead of lists
tokenized_datasets.set_format("torch")

In [ ]:
# Create a smaller subset of the dataset to speed up the fine-tuning 
# You can increase the size of the subset or use the full dataset for better results but it will take longer
# Here, we are using 1000 examples each for training and evaluation
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [6]:
# Import the DataLoader class from PyTorch's torch.utils.data module
# DataLoader allows efficient batching, shuffling, and loading of data for model training and evaluation
from torch.utils.data import DataLoader

# Create a DataLoader for the training dataset
# small_train_dataset is assumed to be a subset of your tokenized training data
# shuffle=True shuffles the data at each epoch to improve model generalization
# batch_size=8 specifies that each batch will contain 8 samples
train_dataloader = DataLoader(small_train_dataset, shuffle=True, batch_size=8)

# Create a DataLoader for the evaluation dataset
# small_eval_dataset is assumed to be a subset of your tokenized evaluation data
# shuffle is not used for evaluation data to maintain consistency in data order
# batch_size=8 specifies that each batch will contain 8 samples
eval_dataloader = DataLoader(small_eval_dataset, batch_size=8)

In [ ]:
# Import the AutoModelForSequenceClassification class from the transformers library
# This class allows loading a pre-trained transformer model specifically for classification tasks
from transformers import AutoModelForSequenceClassification

# Load a pre-trained BERT model ('google-bert/bert-base-cased') for sequence classification
# The num_labels parameter specifies the number of output labels/classes for classification
# Here, num_labels=5 indicates that the model is being fine-tuned for a task with five classes (e.g., a five-class classification problem)
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-cased", num_labels=5)

/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <367D4265-B20F-34BD-94EB-4F3EE47C385B> /Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/torchvision/image.so
  Reason: tried: '/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/mr.engelschion/anaconda3/envs/assignment3/lib/python3.12/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/mr.engelschion/anaconda3/envs/assignment3/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image 

In [ ]:
# Import the AdamW optimizer from PyTorch's torch.optim module
# AdamW is a variant of the Adam optimizer with weight decay, commonly used for fine-tuning transformers
from torch.optim import AdamW

# Create an optimizer for fine-tuning the model
# optimizer updates the model parameters during training to minimize the loss function
# model.parameters() specifies which parameters (weights and biases) to optimize
# lr=5e-5 sets the learning rate for the optimizer, controlling the step size at each update
optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
# Import the get_scheduler function from the transformers library
# This function allows the creation of different learning rate schedulers
from transformers import get_scheduler

# Define the number of epochs (full passes through the training data)
num_epochs = 3

# Calculate the total number of training steps (batches)
# This is the product of the number of epochs and the number of batches in the training DataLoader
num_training_steps = num_epochs * len(train_dataloader)

# Create a learning rate scheduler
# This scheduler linearly decreases the learning rate from its initial value (set in the optimizer) to zero over the course of training
# name="linear" specifies a linear learning rate decay schedule
# optimizer specifies the optimizer to update (AdamW in this case)
# num_warmup_steps=0 means there is no warmup period where the learning rate increases from a small value to the initial value
# num_training_steps specifies the total number of steps for the decay schedule
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [10]:
# Check if a GPU (CUDA) is available and set the device accordingly
# torch.device("cuda") selects a GPU if available for faster computation
# If a GPU is not available, it defaults to using the CPU for computation
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Move the model's parameters to the specified device (GPU or CPU)
# This ensures that all model computations happen on the chosen device
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Training loop

In [11]:
# Note: on my machine, the following code block took about 30 minutes to run

# Import tqdm for creating a progress bar to visually track training progress
# tqdm.auto automatically chooses the appropriate version (e.g., Jupyter-friendly) for display
from tqdm.auto import tqdm

# Create a progress bar for the total number of training steps
# This will visually track the training progress over all epochs and batches
progress_bar = tqdm(range(num_training_steps))

# Set the model to training mode
# This activates behaviors like dropout and batch normalization in training mode
model.train()

# Loop over the number of epochs (full passes through the dataset)
for epoch in range(num_epochs):
    # Loop over each batch in the training DataLoader
    for batch in train_dataloader:
        # Move each batch of data to the specified device (CPU or GPU)
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass: Pass the input data through the model to obtain outputs
        outputs = model(**batch)

        # Compute the loss (automatically computed based on the specified loss function for the model)
        loss = outputs.loss

        # Backpropagation: Compute gradients for model parameters with respect to the loss
        loss.backward()

        # Update model parameters using the optimizer based on computed gradients
        optimizer.step()

        # Update the learning rate using the learning rate scheduler
        lr_scheduler.step()

        # Reset the gradients of the model parameters to zero
        # This prevents the accumulation of gradients from multiple backward passes
        optimizer.zero_grad()

        # Update the progress bar to reflect the completion of another training step
        progress_bar.update(1)


  3%|▎         | 12/375 [04:21<2:17:24, 22.71s/it]

KeyboardInterrupt: 

In [ ]:
# Import the evaluate library to facilitate the computation of metrics
import evaluate

# Load the accuracy metric for evaluation
# This metric will be used to evaluate the model's performance on the evaluation dataset
metric = evaluate.load("accuracy")

# Set the model to evaluation mode
# This deactivates training-specific behaviors such as dropout for more consistent predictions
model.eval()

# Loop over each batch in the evaluation DataLoader
for batch in eval_dataloader:
    # Move the batch of data to the specified device (CPU or GPU) for computation
    batch = {k: v.to(device) for k, v in batch.items()}

    # Disable gradient calculation during evaluation to save memory and computation
    # This ensures no gradients are computed, as they are not needed during evaluation
    with torch.no_grad():
        # Perform a forward pass through the model to obtain outputs (predictions)
        outputs = model(**batch)

    # Extract logits (raw output scores) from the model's output
    logits = outputs.logits

    # Convert the logits to predicted class labels
    # torch.argmax selects the index of the maximum value along the specified dimension (dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    # Add the batch of predictions and reference (true) labels to the metric for computation
    metric.add_batch(predictions=predictions, references=batch["labels"])

# Compute and return the final accuracy metric based on all evaluated batches
metric.compute()


{'accuracy': 0.588}

## Make a prediction using a new text example

In [ ]:
# Function to make predictions for input text
def predict_class(input_text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        input_text,
        return_tensors="pt",        # Return PyTorch tensors
        padding="max_length",       # Pad to the maximum length used during training
        truncation=True,            # Truncate if the input exceeds the maximum length
        max_length=128              # Ensure this matches the max length used during training
    )
    
    # Move input tensors to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Put the model in evaluation mode
    model.eval()
    
    # Make predictions without computing gradients
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits (raw scores) from the model's output
    logits = outputs.logits
    
    # Convert logits to predicted class (taking the index of the max logit)
    predicted_class = torch.argmax(logits, dim=-1).item()
    
    return predicted_class

# Example usage
input_text = "The food at this restaurant was absolutely amazing!"
predicted_class = predict_class(input_text, model, tokenizer, device)

# Display the predicted class, 0 to 4, based on the input text 4 being the highest rating
print(f"The predicted class for the input text is: {predicted_class}")


The predicted class for the input text is: 4


# TASK 2:

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import re
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

class SOTUDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def load_sotu_speeches():
    """Load all SOTU speeches from the directory structure"""
    speeches = []
    try:
        with open('paste.txt', 'r', encoding='utf8') as f:
            file_list_content = f.read()
        speech_files = file_list_content.strip().split('\n')
        print(f"Number of files listed: {len(speech_files)}")
        
        for filepath in speech_files:
            try:
                with open(filepath, 'r', encoding='utf8') as f:
                    content = f.read()
                
                speech_text = re.sub(
                    r'^.*?(Fellow[- ]Citizens|Fellow Citizens).*?\n\n', '', 
                    content, flags=re.DOTALL
                )
                speech_text = re.sub(r'\n\s*\n', '\n\n', speech_text)
                
                if speech_text.strip():
                    year_match = re.search(r'(\d{4})\.txt$', filepath)
                    president_match = re.search(r'sotu/([A-Za-z]+)_', filepath)
                    
                    if year_match and president_match:
                        year = int(year_match.group(1))
                        president = president_match.group(1)
                        speeches.append({
                            'text': speech_text,
                            'year': year,
                            'president': president
                        })
            except Exception as e:
                print(f"Error loading {filepath}: {str(e)}")
    except Exception as e:
        print(f"Error reading file list: {str(e)}")
        return []

    speeches.sort(key=lambda x: x['year'])
    return speeches

def preprocess_text(text):
    """Clean and preprocess text"""
    text = re.sub(r'[^\w\s]', '', text)
    text = ' '.join(text.split())
    return text

def prepare_data(speeches, min_paragraph_length=50, max_paragraphs=1000):
    """Prepare SOTU speeches data with metadata and limit total paragraphs"""
    paragraphs = []
    metadata = []
    count = 0
    
    print("Starting data preparation...")
    for speech in speeches:
        if count >= max_paragraphs:
            break
            
        speech_paragraphs = speech['text'].split('\n\n')
        
        for para in speech_paragraphs:
            if count >= max_paragraphs:
                break
                
            processed_para = preprocess_text(para)
            if len(processed_para.split()) >= min_paragraph_length:
                paragraphs.append(processed_para)
                metadata.append({
                    'year': speech['year'],
                    'president': speech['president']
                })
                count += 1
    
    # Simplified sentiment lexicon
    positive_words = [
        'progress', 'success', 'growth', 'prosperity', 'victory', 'peace', 
        'achievement', 'improve', 'strength', 'opportunity', 'hope'
    ]
    
    negative_words = [
        'crisis', 'conflict', 'failure', 'deficit', 'war', 'problem', 'threat',
        'poverty', 'decline', 'fear', 'danger'
    ]
    
    # Label paragraphs
    labels = []
    for para in paragraphs:
        para_lower = para.lower()
        pos_count = sum(1 for word in positive_words if word in para_lower)
        neg_count = sum(1 for word in negative_words if word in para_lower)
        
        if pos_count > neg_count:
            labels.append(2)  # Positive
        elif neg_count > pos_count:
            labels.append(0)  # Negative
        else:
            labels.append(1)  # Neutral

    print(f"Total paragraphs processed: {len(paragraphs)}")
    return paragraphs, labels, metadata

def evaluate_baseline_model(texts, labels):
    """Evaluate a simple dictionary-based baseline model"""
    try:
        nltk.data.find('sentiment/vader_lexicon.zip')
    except LookupError:
        nltk.download('vader_lexicon')
    
    sia = SentimentIntensityAnalyzer()
    predictions = []
    
    for text in tqdm(texts, desc="Evaluating baseline"):
        scores = sia.polarity_scores(text)
        if scores['compound'] > 0.1:
            predictions.append(2)  # Positive
        elif scores['compound'] < -0.1:
            predictions.append(0)  # Negative
        else:
            predictions.append(1)  # Neutral
    
    accuracy = np.mean(np.array(predictions) == np.array(labels))
    return accuracy

def evaluate_untrained_model(eval_dataloader, model, device):
    """Evaluate model without any training"""
    model.eval()
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc="Evaluating untrained model"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            
            outputs = model(input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
            actual_labels.extend(labels.numpy())
    
    accuracy = np.mean(np.array(predictions) == np.array(actual_labels))
    return accuracy

def train_model(train_dataloader, model, optimizer, device):
    """Train model for one epoch"""
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_dataloader)

def evaluate_model(eval_dataloader, model, device):
    """Evaluate model"""
    model.eval()
    total_loss = 0
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            
            predictions.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
            actual_labels.extend(labels.cpu().numpy())
    
    accuracy = np.mean(np.array(predictions) == np.array(actual_labels))
    return total_loss / len(eval_dataloader), accuracy

def main():
    # Load speeches
    speeches = load_sotu_speeches()
    if not speeches:
        print("No speeches loaded. Exiting.")
        return
    
    # Prepare data with limit
    paragraphs, labels, metadata = prepare_data(speeches, max_paragraphs=1000)
    
    if len(paragraphs) == 0 or len(labels) == 0:
        print("No paragraphs or labels to process. Exiting.")
        return

    # Split data
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        paragraphs, labels, test_size=0.2, random_state=42
    )
    
    # Initialize model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=3
    )
    
    # Initialize untrained model for baseline
    untrained_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=3
    )
    
    # Create datasets
    train_dataset = SOTUDataset(train_texts, train_labels, tokenizer)
    val_dataset = SOTUDataset(val_texts, val_labels, tokenizer)
    
    # Create dataloaders
    train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=4)
    
    # Setup training
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    model.to(device)
    untrained_model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # Get baseline performances
    print("\nBaseline Evaluations:")
    print("====================")
    
    # Dictionary-based baseline
    baseline_accuracy = evaluate_baseline_model(val_texts, val_labels)
    print(f"Dictionary-based baseline accuracy: {baseline_accuracy:.4f}")
    
    # Untrained model baseline
    untrained_accuracy = evaluate_untrained_model(val_dataloader, untrained_model, device)
    print(f"Untrained model accuracy: {untrained_accuracy:.4f}")
    
    # Training loop with early stopping
    num_epochs = 2
    best_val_accuracy = 0
    patience = 2
    patience_counter = 0
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        train_loss = train_model(train_dataloader, model, optimizer, device)
        val_loss, val_accuracy = evaluate_model(val_dataloader, model, device)
        
        print(f"Training Loss: {train_loss:.4f}")
        print(f"Validation Loss: {val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy:.4f}")
        
        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            model.save_pretrained("./best_model_checkpoint")
            tokenizer.save_pretrained("./best_model_checkpoint")
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Early stopping
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    # Print final performance comparison
    print("\nFinal Performance Metrics:")
    print("=========================")
    print(f"Dictionary-based baseline accuracy: {baseline_accuracy:.4f}")
    print(f"Untrained model accuracy: {untrained_accuracy:.4f}")
    print(f"Fine-tuned model accuracy: {best_val_accuracy:.4f}")
    
    # Calculate improvements
    improvement_over_baseline = (best_val_accuracy - baseline_accuracy) / baseline_accuracy * 100
    improvement_over_untrained = (best_val_accuracy - untrained_accuracy) / untrained_accuracy * 100
    
    print(f"\nImprovements:")
    print(f"Improvement over dictionary baseline: {improvement_over_baseline:.2f}%")
    print(f"Improvement over untrained model: {improvement_over_untrained:.2f}%")
    
    # Print model architecture and training details
    print("\nModel Architecture:")
    print(f"Model name: {model.config.model_type}")
    print(f"Hidden layers: {model.config.n_layers}")
    print(f"Attention heads: {model.config.n_heads}")
    print(f"Hidden size: {model.config.hidden_size}")
    
    # Print training configuration
    print("\nTraining Configuration:")
    print(f"Batch size: {train_dataloader.batch_size}")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']}")
    print(f"Max sequence length: {train_dataset.max_length}")
    print(f"Device used: {device}")
    
    # Print data statistics
    label_dist = np.bincount(labels)
    print("\nLabel Distribution:")
    print(f"Negative (0): {label_dist[0]} ({label_dist[0]/len(labels)*100:.2f}%)")
    print(f"Neutral  (1): {label_dist[1]} ({label_dist[1]/len(labels)*100:.2f}%)")
    print(f"Positive (2): {label_dist[2]} ({label_dist[2]/len(labels)*100:.2f}%)")

if __name__ == "__main__":
    main()

Number of files listed: 223
Starting data preparation...
Total paragraphs processed: 1000


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cpu

Baseline Evaluations:


Evaluating baseline: 100%|██████████| 200/200 [00:00<00:00, 1886.81it/s]


Dictionary-based baseline accuracy: 0.3250


Evaluating untrained model: 100%|██████████| 50/50 [00:04<00:00, 10.03it/s]


Untrained model accuracy: 0.2650

Epoch 1/2


Evaluating: 100%|██████████| 50/50 [00:03<00:00, 12.98it/s]


Training Loss: 1.0476
Validation Loss: 0.9576
Validation Accuracy: 0.5550

Epoch 2/2


Evaluating: 100%|██████████| 50/50 [00:04<00:00, 11.13it/s]


Training Loss: 0.7972
Validation Loss: 0.7855
Validation Accuracy: 0.7000

Final Performance Metrics:
Dictionary-based baseline accuracy: 0.3250
Untrained model accuracy: 0.2650
Fine-tuned model accuracy: 0.7000

Improvements:
Improvement over dictionary baseline: 115.38%
Improvement over untrained model: 164.15%

Model Architecture:
Model name: distilbert
Hidden layers: 6
Attention heads: 12
Hidden size: 768

Training Configuration:
Batch size: 4
Learning rate: 2e-05
Max sequence length: 128
Device used: cpu

Label Distribution:
Negative (0): 217 (21.70%)
Neutral  (1): 492 (49.20%)
Positive (2): 291 (29.10%)
